# Create Heineken Prizes Awards

Creates OpenAlex award rows from the official Heineken Prizes WordPress API.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/heineken_prizes_to_s3.py` to fetch the official WordPress portfolio records, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data sources:**
- Laureate list page: `https://www.heinekenprizes.org/list-of-laureates/`
- Portfolio API: `https://www.heinekenprizes.org/wp-json/wp/v2/avada_portfolio`
- Heineken Prizes amount/rules page: `https://www.heinekenprizes.org/heineken-prizes/`
- Heineken Young Scientists Awards amount/rules page: `https://www.heinekenprizes.org/heineken-young-scientists-awards/`

**S3 location:** `s3a://openalex-ingest/awards/heineken_prizes/heineken_prizes_laureates.parquet`

**Local full-source validation on 2026-05-27:** 144 official laureate rows, 1964-2024. Coverage: 100% title/year/recipient/scheme/research-field/description/amount/currency/landing-url. Rows split into 108 Heineken Prize laureates and 36 Heineken Young Scientists Award laureates.

**OpenAlex funder mapping:**
- Chosen funder_id: 4320320934
- Display name: Koninklijke Nederlandse Akademie van Wetenschappen
- Country: NL
- DOI/ROR: not present on the OpenAlex funder row
- Rationale: Heineken Prizes are awarded under the responsibility of the Royal Netherlands Academy of Arts and Sciences (KNAW). The source pages name the Alfred Heineken funds as funders, but OpenAlex has no separate Heineken prize-fund funder row; KNAW is the awarding/selection body and is the auditable OpenAlex funder.

**Mapping summary:** One OpenAlex award row per official laureate. `amount`/`currency` are populated from official program rules: USD 250,000 for scientific Heineken Prizes, EUR 100,000 for the art prize, and EUR 15,000 for Heineken Young Scientists Awards. The archive does not publish structured affiliation fields, so laureate affiliations remain NULL rather than inferred from prose.


## Step 1: Load Raw Parquet

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.heineken_prizes_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/heineken_prizes/heineken_prizes_laureates.parquet`;


In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-27 found 144 rows.
SELECT COUNT(*) as total_heineken_prize_laureates
FROM openalex.awards.heineken_prizes_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.heineken_prizes_raw;


In [ ]:
%sql
SELECT funder_award_id, display_name, source_year, recipient_name, award_family,
       funder_scheme, research_field, amount, currency, landing_page_url
FROM openalex.awards.heineken_prizes_raw
ORDER BY TRY_CAST(source_year AS INT) DESC, funder_scheme, recipient_name
LIMIT 20;


## Step 1.5: Source, Money, and Key Scans

In [ ]:
%sql
SELECT column_name
FROM (DESCRIBE openalex.awards.heineken_prizes_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant|award|currency';


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS rows_with_title,
    COUNT(source_year) AS rows_with_year,
    COUNT(recipient_name) AS rows_with_recipient,
    COUNT(funder_scheme) AS rows_with_scheme,
    COUNT(research_field) AS rows_with_research_field,
    COUNT(description) AS rows_with_description,
    COUNT(amount) AS rows_with_amount,
    COUNT(currency) AS rows_with_currency,
    COUNT(landing_page_url) AS rows_with_url,
    MIN(TRY_CAST(source_year AS INT)) AS min_year,
    MAX(TRY_CAST(source_year AS INT)) AS max_year,
    SUM(TRY_CAST(amount AS DOUBLE)) AS total_nominal_amount
FROM openalex.awards.heineken_prizes_raw;


In [ ]:
%sql
SELECT award_family, funder_scheme, research_field, COUNT(*) AS rows,
       MIN(TRY_CAST(source_year AS INT)) AS min_year,
       MAX(TRY_CAST(source_year AS INT)) AS max_year
FROM openalex.awards.heineken_prizes_raw
GROUP BY award_family, funder_scheme, research_field
ORDER BY award_family, funder_scheme;


In [ ]:
%sql
SELECT currency, TRY_CAST(amount AS DOUBLE) AS amount, COUNT(*) AS rows
FROM openalex.awards.heineken_prizes_raw
GROUP BY currency, TRY_CAST(amount AS DOUBLE)
ORDER BY currency, amount;


In [ ]:
%sql
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.heineken_prizes_raw
GROUP BY funder_award_id
HAVING COUNT(*) > 1;


## Step 1.6: Funder Existence Fail-Fast

In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for KNAW F4320320934'
) AS funder_row_exists
FROM openalex.common.funder
WHERE funder_id = 4320320934;


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320320934;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.heineken_prizes_awards
USING delta
AS
WITH
funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320320934
),
raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date,
        TRY_CAST(source_year AS INT) AS parsed_year
    FROM openalex.awards.heineken_prizes_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,
        TRIM(g.display_name) as display_name,
        NULLIF(TRIM(g.description), '') as description,
        f.funder_id,
        g.native_award_id as funder_award_id,
        g.parsed_amount as amount,
        NULLIF(TRIM(g.currency), '') as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,
        'prize' as funding_type,
        COALESCE(NULLIF(TRIM(g.funder_scheme), ''), 'Heineken Prizes') as funder_scheme,
        'heineken_prizes_wp' as provenance,
        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        COALESCE(YEAR(g.parsed_start_date), g.parsed_year) as start_year,
        COALESCE(YEAR(g.parsed_end_date), g.parsed_year) as end_year,
        struct(
            NULLIF(TRIM(g.given_name), '') as given_name,
            NULLIF(TRIM(g.family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                CAST(NULL AS STRING) as name,
                CAST(NULL AS STRING) as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM raw_prepared g
    CROSS JOIN funder_resolved f
)
SELECT * FROM awards_transformed;


## Step 3: Delete Previous Source Rows and Insert Priority 128

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'heineken_prizes_wp' AND priority = 128;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    128 as priority
FROM openalex.awards.heineken_prizes_awards;


## Verification

In [ ]:
%sql
SELECT COUNT(*) as total_heineken_awards
FROM openalex.awards.heineken_prizes_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.heineken_prizes_awards;


In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_year,
       lead_investigator.given_name AS given_name,
       lead_investigator.family_name AS family_name,
       landing_page_url
FROM openalex.awards.heineken_prizes_awards
ORDER BY start_year DESC, funder_scheme, display_name
LIMIT 20;


In [ ]:
%sql
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT id) AS distinct_openalex_ids,
       COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.heineken_prizes_awards;


In [ ]:
%sql
SELECT funder.display_name, funder_id, COUNT(*) AS cnt
FROM openalex.awards.heineken_prizes_awards
GROUP BY funder.display_name, funder_id;


In [ ]:
%sql
SELECT COUNT(*) as total, COUNT(description) as has_description, COUNT(amount) as has_amount,
       COUNT(currency) as has_currency, COUNT(lead_investigator) as has_lead,
       COUNT(lead_investigator.affiliation.name) as has_affiliation
FROM openalex.awards.heineken_prizes_awards;


In [ ]:
%sql
SELECT currency, amount, COUNT(*) AS rows
FROM openalex.awards.heineken_prizes_awards
GROUP BY currency, amount
ORDER BY currency, amount;


In [ ]:
%sql
SELECT start_year, funder_scheme, COUNT(*) AS rows
FROM openalex.awards.heineken_prizes_awards
GROUP BY start_year, funder_scheme
ORDER BY start_year DESC, funder_scheme;


In [ ]:
%sql
SELECT priority, provenance, COUNT(*) as cnt, COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'heineken_prizes_wp' AND priority = 128
GROUP BY priority, provenance;
